# KV cache: prefill and decode

How much work does caching save, and how much memory does the cache need?
Run commands are in [README.md](README.md). This notebook only reads results.
Three repeats compare the same tiny model, device and fixed continuation tokens.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from llm_systems_lab.inference_results import load_inference_results

root = Path.cwd()
if root.name == "04_kv_cache":
    root = root.parent.parent
results_dir = root / "experiments/04_kv_cache/results"


In [ ]:
runs, summary = load_inference_results(sorted(results_dir.glob("repeat-*/*.json")))
columns = ["mode", "prefill_mean_ms", "prefill_median_ms",
           "decode_mean_ms", "decode_median_ms", "tokens_per_second", "cache_MiB"]
display(summary[columns].round(3))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
summary[["prefill_mean_ms", "prefill_median_ms"]].plot.bar(ax=axes[0], rot=35)
axes[0].set(title="Prefill", ylabel="Time [ms]", xlabel="")
for name, result in runs.items():
    samples = pd.DataFrame(result["decode"]["samples_seconds"]) * 1000
    lengths = result["decode"]["context_lengths"]
    color = "tab:orange" if result["mode"] == "kv_cache" else "tab:blue"
    axes[1].plot(lengths, samples.mean(), color=color, label=name)
    axes[1].plot(lengths, samples.median(), "--", color=color)
axes[1].set(title="Decode: mean (solid), median (dashed)",
            xlabel="Context length [tokens]", ylabel="Time per batch step [ms]")
axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()


Prefill processes the prompt. Each decode call consumes one fixed continuation token.
Without cache, the model recomputes the full prefix. Tokens/s counts the whole batch.

Cache MiB is the logical size of the final K/V tensors, excluding weights and temporaries.
It is not peak memory. A zero in no-cache mode does not mean zero memory use.

The cache still grows via torch.cat, which copies the earlier K/V. Fixed buffers are a later experiment.

Observed: cached decode is not consistently faster across these three tiny MPS repeats.
Both modes vary between runs. We have not profiled what causes the timing differences.
Correctness and less recomputation do not automatically imply a wall-clock speedup.
